# Entender `metrics.py` paso a paso

Este notebook explica que devuelven las funciones de `src/retail_forecasting/evaluation/metrics.py` usando una tabla sintetica pequena. La idea es ver los resultados sin entrenar modelos ni depender del dataset real.

## Que vas a ver

- que columnas necesita una tabla de predicciones;
- que devuelve `pinball_loss()`;
- que devuelven los dos DataFrames de `summarize_predictions()`;
- que devuelve `summarize_costs()`;
- como interpretar esos resultados en este problema de forecasting e inventario.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
warnings.filterwarnings("ignore")

from retail_forecasting.config import InventoryConfig
from retail_forecasting.evaluation.metrics import pinball_loss, summarize_costs, summarize_predictions
from retail_forecasting.inventory.newsvendor import attach_inventory_costs, choose_order_quantity


## 1. Construimos una tabla de predicciones pequena

En el pipeline real, `metrics.py` no entrena modelos. Recibe una tabla ya generada por el backtesting. En esa tabla:

- `y_true` es la demanda real acumulada durante el lead time;
- `y_pred` es la prediccion puntual del modelo;
- `q_0_1`, `q_0_5`, `q_0_9` son predicciones cuantiles, si el modelo las produce;
- `model_name`, `backend_name` y `fold_id` permiten agrupar resultados;
- las columnas de inventario vienen de `newsvendor.py` y luego las resume `summarize_costs()`.


In [2]:
def make_example_predictions() -> pd.DataFrame:
    examples = [
        {"fold_id": 0, "date": "2025-03-01", "series_id": "store_1_product_101", "y_true": 42.0, "naive_pred": 48.0, "boost_pred": 44.0},
        {"fold_id": 0, "date": "2025-03-01", "series_id": "store_2_product_102", "y_true": 55.0, "naive_pred": 50.0, "boost_pred": 53.0},
        {"fold_id": 0, "date": "2025-03-02", "series_id": "store_1_product_101", "y_true": 48.0, "naive_pred": 52.0, "boost_pred": 49.0},
        {"fold_id": 0, "date": "2025-03-02", "series_id": "store_2_product_102", "y_true": 63.0, "naive_pred": 55.0, "boost_pred": 60.0},
        {"fold_id": 1, "date": "2025-03-08", "series_id": "store_1_product_101", "y_true": 45.0, "naive_pred": 47.0, "boost_pred": 46.0},
        {"fold_id": 1, "date": "2025-03-08", "series_id": "store_2_product_102", "y_true": 58.0, "naive_pred": 51.0, "boost_pred": 56.0},
        {"fold_id": 1, "date": "2025-03-09", "series_id": "store_1_product_101", "y_true": 51.0, "naive_pred": 54.0, "boost_pred": 52.0},
        {"fold_id": 1, "date": "2025-03-09", "series_id": "store_2_product_102", "y_true": 67.0, "naive_pred": 59.0, "boost_pred": 64.0},
    ]

    rows = []
    for row in examples:
        common = {
            "fold_id": row["fold_id"],
            "date": pd.Timestamp(row["date"]),
            "series_id": row["series_id"],
            "y_true": row["y_true"],
        }

        rows.append(
            {
                **common,
                "model_name": "seasonal_naive",
                "backend_name": "heuristic",
                "y_pred": row["naive_pred"],
                "q_0_1": np.nan,
                "q_0_5": np.nan,
                "q_0_9": np.nan,
            }
        )

        rows.append(
            {
                **common,
                "model_name": "auto_boosting",
                "backend_name": "sklearn_hist_gradient_boosting",
                "y_pred": row["boost_pred"],
                "q_0_1": row["boost_pred"] - 8.0,
                "q_0_5": row["boost_pred"],
                "q_0_9": row["boost_pred"] + 10.0,
            }
        )

    return pd.DataFrame(rows)


predictions = make_example_predictions()
display(predictions)


,fold_id,date,series_id,y_true,model_name,backend_name,y_pred,q_0_1,q_0_5,q_0_9
0,0,2025-03-01,store_1_product_101,42.0,seasonal_naive,heuristic,48.0,NaN,NaN,NaN
1,0,2025-03-01,store_1_product_101,42.0,auto_boosting,sklearn_hist_gradient_boosting,44.0,36.0,44.0,54.0
2,0,2025-03-01,store_2_product_102,55.0,seasonal_naive,heuristic,50.0,NaN,NaN,NaN
3,0,2025-03-01,store_2_product_102,55.0,auto_boosting,sklearn_hist_gradient_boosting,53.0,45.0,53.0,63.0
4,0,2025-03-02,store_1_product_101,48.0,seasonal_naive,heuristic,52.0,NaN,NaN,NaN
5,0,2025-03-02,store_1_product_101,48.0,auto_boosting,sklearn_hist_gradient_boosting,49.0,41.0,49.0,59.0
6,0,2025-03-02,store_2_product_102,63.0,seasonal_naive,heuristic,55.0,NaN,NaN,NaN
7,0,2025-03-02,store_2_product_102,63.0,auto_boosting,sklearn_hist_gradient_boosting,60.0,52.0,60.0,70.0
8,1,2025-03-08,store_1_product_101,45.0,seasonal_naive,heuristic,47.0,NaN,NaN,NaN
9,1,2025-03-08,store_1_product_101,45.0,auto_boosting,sklearn_hist_gradient_boosting,46.0,38.0,46.0,56.0


## 2. Anadimos la decision de inventario y sus costes

`summarize_costs()` espera que la tabla ya tenga columnas como `order_quantity`, `overstock_units`, `stockout_units` y `total_cost`. En el pipeline real esas columnas se calculan antes, en `inventory/newsvendor.py`.

Aqui usamos la misma logica real para que el ejemplo sea fiel al proyecto:

- el baseline no tiene cuantiles, asi que pide segun `y_pred`;
- el boosting tiene cuantiles, asi que interpola el cuantil optimo segun los costes de overstock y stockout.


In [3]:
inventory_config = InventoryConfig(overstock_cost=1.0, stockout_cost=4.0)
quantile_columns = ["q_0_1", "q_0_5", "q_0_9"]
quantile_levels = [0.1, 0.5, 0.9]

predictions = predictions.copy()
predictions["order_quantity"] = np.nan

for model_name, subset in predictions.groupby("model_name", dropna=False):
    has_quantiles = subset.loc[:, quantile_columns].notna().any().any()
    predictions.loc[subset.index, "order_quantity"] = choose_order_quantity(
        predictions=subset,
        inventory_config=inventory_config,
        quantile_columns=quantile_columns if has_quantiles else [],
        quantile_levels=quantile_levels if has_quantiles else [],
    )

predictions = attach_inventory_costs(predictions, inventory_config)

columns_to_show = [
    "fold_id",
    "date",
    "series_id",
    "model_name",
    "backend_name",
    "y_true",
    "y_pred",
    "q_0_1",
    "q_0_5",
    "q_0_9",
    "order_quantity",
    "overstock_units",
    "stockout_units",
    "total_cost",
]

display(predictions.loc[:, columns_to_show])


,fold_id,date,series_id,model_name,backend_name,y_true,y_pred,q_0_1,q_0_5,q_0_9,order_quantity,overstock_units,stockout_units,total_cost
0,0,2025-03-01,store_1_product_101,seasonal_naive,heuristic,42.0,48.0,NaN,NaN,NaN,48.0,6.0,0.0,6.0
1,0,2025-03-01,store_1_product_101,auto_boosting,sklearn_hist_gradient_boosting,42.0,44.0,36.0,44.0,54.0,51.5,9.5,0.0,9.5
2,0,2025-03-01,store_2_product_102,seasonal_naive,heuristic,55.0,50.0,NaN,NaN,NaN,50.0,0.0,5.0,20.0
3,0,2025-03-01,store_2_product_102,auto_boosting,sklearn_hist_gradient_boosting,55.0,53.0,45.0,53.0,63.0,60.5,5.5,0.0,5.5
4,0,2025-03-02,store_1_product_101,seasonal_naive,heuristic,48.0,52.0,NaN,NaN,NaN,52.0,4.0,0.0,4.0
5,0,2025-03-02,store_1_product_101,auto_boosting,sklearn_hist_gradient_boosting,48.0,49.0,41.0,49.0,59.0,56.5,8.5,0.0,8.5
6,0,2025-03-02,store_2_product_102,seasonal_naive,heuristic,63.0,55.0,NaN,NaN,NaN,55.0,0.0,8.0,32.0
7,0,2025-03-02,store_2_product_102,auto_boosting,sklearn_hist_gradient_boosting,63.0,60.0,52.0,60.0,70.0,67.5,4.5,0.0,4.5
8,1,2025-03-08,store_1_product_101,seasonal_naive,heuristic,45.0,47.0,NaN,NaN,NaN,47.0,2.0,0.0,2.0
9,1,2025-03-08,store_1_product_101,auto_boosting,sklearn_hist_gradient_boosting,45.0,46.0,38.0,46.0,56.0,53.5,8.5,0.0,8.5


## 3. `pinball_loss()` devuelve un numero float

`pinball_loss(actual, predicted, quantile)` mide la calidad de un cuantil. Penaliza de forma asimetrica:

- para un cuantil alto, como 0.9, subestimar la demanda penaliza mas;
- para un cuantil bajo, como 0.1, sobreestimar la demanda penaliza mas.

En este problema se usa para evaluar si los cuantiles `q_0_1`, `q_0_5` y `q_0_9` estan bien calibrados.


In [4]:
pinball_examples = []
for case_name, actual_value, predicted_value in [
    ("subestima demanda", 100.0, 90.0),
    ("sobrestima demanda", 100.0, 110.0),
]:
    for quantile in [0.1, 0.5, 0.9]:
        pinball_examples.append(
            {
                "case": case_name,
                "actual": actual_value,
                "predicted": predicted_value,
                "quantile": quantile,
                "pinball_loss": pinball_loss(
                    pd.Series([actual_value]),
                    pd.Series([predicted_value]),
                    quantile,
                ),
            }
        )

display(pd.DataFrame(pinball_examples))


,case,actual,predicted,quantile,pinball_loss
0,subestima demanda,100.0,90.0,0.1,1.0
1,subestima demanda,100.0,90.0,0.5,5.0
2,subestima demanda,100.0,90.0,0.9,9.0
3,sobrestima demanda,100.0,110.0,0.1,9.0
4,sobrestima demanda,100.0,110.0,0.5,5.0
5,sobrestima demanda,100.0,110.0,0.9,1.0


## 4. `summarize_predictions()` devuelve dos DataFrames

La funcion devuelve una tupla:

```python
metrics_summary, fold_metrics = summarize_predictions(predictions)
```

- `metrics_summary`: una fila por combinacion `model_name` + `backend_name`, agregando todos los folds;
- `fold_metrics`: una fila por `fold_id` + `model_name` + `backend_name`, para ver estabilidad por fold.


In [5]:
metrics_summary, fold_metrics = summarize_predictions(predictions)

print("metrics_summary: resumen global por modelo/backend")
display(metrics_summary)

print("fold_metrics: resumen por fold temporal y modelo/backend")
display(fold_metrics)


metrics_summary: resumen global por modelo/backend


,model_name,backend_name,observations,mae,rmse,pinball_q_0_1,pinball_q_0_5,pinball_q_0_9,coverage_q_0_1_q_0_9
0,auto_boosting,sklearn_hist_gradient_boosting,8,1.875,2.03101,0.8625,0.9375,0.9375,1.0
1,seasonal_naive,heuristic,8,5.375,5.77711,NaN,NaN,NaN,NaN


fold_metrics: resumen por fold temporal y modelo/backend


,model_name,backend_name,observations,mae,rmse,pinball_q_0_1,pinball_q_0_5,pinball_q_0_9,coverage_q_0_1_q_0_9,fold_id
0,auto_boosting,sklearn_hist_gradient_boosting,4,2.00,2.121320,0.850,1.000,0.950,1.0,0
1,seasonal_naive,heuristic,4,5.75,5.937171,NaN,NaN,NaN,NaN,0
2,auto_boosting,sklearn_hist_gradient_boosting,4,1.75,1.936492,0.875,0.875,0.925,1.0,1
3,seasonal_naive,heuristic,4,5.00,5.612486,NaN,NaN,NaN,NaN,1


## 5. Comprobamos MAE y RMSE a mano

`metrics.py` calcula los errores como:

```python
errors = y_pred - y_true
mae = mean(abs(errors))
rmse = sqrt(mean(errors ** 2))
```

En forecasting de demanda:

- `mae` dice cuantas unidades se falla de media;
- `rmse` penaliza mas los errores grandes, por eso suele ser mayor o igual que MAE.


In [6]:
boosting_rows = predictions.loc[predictions["model_name"] == "auto_boosting"].copy()
boosting_rows["error"] = boosting_rows["y_pred"] - boosting_rows["y_true"]
boosting_rows["absolute_error"] = boosting_rows["error"].abs()
boosting_rows["squared_error"] = boosting_rows["error"] ** 2

manual_mae = boosting_rows["absolute_error"].mean()
manual_rmse = np.sqrt(boosting_rows["squared_error"].mean())

display(boosting_rows.loc[:, ["fold_id", "series_id", "y_true", "y_pred", "error", "absolute_error", "squared_error"]])
print("MAE manual auto_boosting:", round(float(manual_mae), 4))
print("RMSE manual auto_boosting:", round(float(manual_rmse), 4))
display(metrics_summary.loc[metrics_summary["model_name"] == "auto_boosting", ["model_name", "mae", "rmse"]])


,fold_id,series_id,y_true,y_pred,error,absolute_error,squared_error
1,0,store_1_product_101,42.0,44.0,2.0,2.0,4.0
3,0,store_2_product_102,55.0,53.0,-2.0,2.0,4.0
5,0,store_1_product_101,48.0,49.0,1.0,1.0,1.0
7,0,store_2_product_102,63.0,60.0,-3.0,3.0,9.0
9,1,store_1_product_101,45.0,46.0,1.0,1.0,1.0
11,1,store_2_product_102,58.0,56.0,-2.0,2.0,4.0
13,1,store_1_product_101,51.0,52.0,1.0,1.0,1.0
15,1,store_2_product_102,67.0,64.0,-3.0,3.0,9.0


MAE manual auto_boosting: 1.875
RMSE manual auto_boosting: 2.031


,model_name,mae,rmse
0,auto_boosting,1.875,2.03101


## 6. Interpretamos las columnas de cuantiles

Si existen columnas que empiezan por `q_` y tienen algun valor no nulo, `summarize_predictions()` anade metricas de pinball loss:

- `pinball_q_0_1` evalua el cuantil bajo;
- `pinball_q_0_5` evalua la mediana;
- `pinball_q_0_9` evalua el cuantil alto.

Tambien calcula `coverage_q_0_1_q_0_9`, que es el porcentaje de observaciones reales que caen dentro del intervalo `[q_0_1, q_0_9]`. En el baseline sale `NaN` porque no produce cuantiles.


In [7]:
coverage_columns = [
    "model_name",
    "backend_name",
    "pinball_q_0_1",
    "pinball_q_0_5",
    "pinball_q_0_9",
    "coverage_q_0_1_q_0_9",
]

display(metrics_summary.loc[:, coverage_columns])

boosting_interval_check = predictions.loc[predictions["model_name"] == "auto_boosting", [
    "series_id",
    "y_true",
    "q_0_1",
    "q_0_9",
]].copy()
boosting_interval_check["inside_interval"] = (
    (boosting_interval_check["y_true"] >= boosting_interval_check["q_0_1"])
    & (boosting_interval_check["y_true"] <= boosting_interval_check["q_0_9"])
)

display(boosting_interval_check)
print("Coverage manual:", boosting_interval_check["inside_interval"].mean())


,model_name,backend_name,pinball_q_0_1,pinball_q_0_5,pinball_q_0_9,coverage_q_0_1_q_0_9
0,auto_boosting,sklearn_hist_gradient_boosting,0.8625,0.9375,0.9375,1.0
1,seasonal_naive,heuristic,NaN,NaN,NaN,NaN


,series_id,y_true,q_0_1,q_0_9,inside_interval
1,store_1_product_101,42.0,36.0,54.0,True
3,store_2_product_102,55.0,45.0,63.0,True
5,store_1_product_101,48.0,41.0,59.0,True
7,store_2_product_102,63.0,52.0,70.0,True
9,store_1_product_101,45.0,38.0,56.0,True
11,store_2_product_102,58.0,48.0,66.0,True
13,store_1_product_101,51.0,44.0,62.0,True
15,store_2_product_102,67.0,56.0,74.0,True


Coverage manual: 1.0


## 7. `summarize_costs()` devuelve un DataFrame de costes

Esta funcion agrupa por `model_name` + `backend_name` y resume la calidad de la decision de inventario, no solo la calidad estadistica del forecast.

Columnas clave:

- `mean_order_quantity`: pedido medio recomendado;
- `total_overstock_units`: unidades que sobran;
- `total_stockout_units`: unidades que faltan;
- `total_overstock_cost`: coste por pedir de mas;
- `total_stockout_cost`: coste por pedir de menos;
- `total_cost`: coste total, ordenado de menor a mayor.


In [8]:
cost_summary = summarize_costs(predictions)
display(cost_summary)


,model_name,backend_name,observations,mean_order_quantity,total_overstock_units,total_stockout_units,total_overstock_cost,total_stockout_cost,total_cost,mean_cost
0,auto_boosting,sklearn_hist_gradient_boosting,8,60.5,55.0,0.0,55.0,0.0,55.0,6.875
1,seasonal_naive,heuristic,8,52.0,15.0,28.0,15.0,112.0,127.0,15.875


## 8. Lectura final para este problema

En este proyecto no basta con mirar solo MAE o RMSE. El objetivo final es tomar mejores decisiones de inventario bajo incertidumbre.

- `summarize_predictions()` responde: que modelo predice mejor la demanda.
- `pinball_loss()` responde: que tan buenos son los cuantiles probabilisticos.
- `coverage_q_0_1_q_0_9` responde: si el intervalo predictivo contiene la demanda real con frecuencia razonable.
- `summarize_costs()` responde: que modelo produce mejores decisiones de pedido segun los costes de overstock y stockout.

El trade-off importante es que un modelo puede tener un error puntual bajo y aun asi generar peores decisiones de inventario si sus cuantiles o su nivel de servicio no estan bien calibrados.
